# DeFoG — Session Init + Marginal Ablation (run at the start of every session)

Restores the full `defog` environment from Google Drive, wires up all paths,
and runs the two-way marginal ablation experiment on QM9.

Run **all cells top-to-bottom** in a single session — do not open the ablation
cells in a separate tab (each tab gets its own VM and a fresh `/content/`).

### Prerequisites
- `01_env_setup.ipynb` has been run at least once (tarball exists on Drive).
- `spminer_motif_marginals.pt` is in `MyDrive/DeFoGColab/data/qm9/`.
- `WANDB_API_KEY` is saved as a Colab Secret (left sidebar → 🔑 Secrets).

### Expected runtime
- Environment restore: ~6–8 min
- Ablation (2 × 100 epochs + test): ~3–5 h on a T4 GPU

In [ ]:
# ── Cell 1: Mount Drive + load W&B API key from Colab Secrets ─────────────────
from google.colab import drive, userdata
import os

drive.mount('/content/drive')

# Reads the key you stored under the name WANDB_API_KEY in Colab Secrets.
# If the secret is missing, training will still work but W&B logging will fail.
try:
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B API key loaded from Colab Secrets.')
except Exception as e:
    print(f'Warning: could not load WANDB_API_KEY ({e}).')
    print('Set it manually: os.environ["WANDB_API_KEY"] = "<your-key>"')

# Suppress matplotlib GUI attempts; graph_tool respects this too.
os.environ['MPLBACKEND'] = 'Agg'

print('Drive mounted.')

In [ ]:
%%bash
# ── Cell 2: Install Miniforge ─────────────────────────────────────────────────
# Miniforge is the conda-forge project's own conda installer.
# It ships pre-configured to use conda-forge only, so no Anaconda
# Terms-of-Service acceptance is ever required (unlike Miniconda ≥ 2024).
# Miniforge itself is not cached — only the defog env tarball is (~30 sec).
set -e
if [ -f /content/miniconda3/bin/conda ]; then
    echo "Conda already installed."
    /content/miniconda3/bin/conda --version
    exit 0
fi
wget -q \
  https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh \
  -O /tmp/miniforge.sh
bash /tmp/miniforge.sh -b -p /content/miniconda3
rm /tmp/miniforge.sh
echo "Miniforge installed."
/content/miniconda3/bin/conda --version

In [ ]:
%%bash
# ── Cell 3: Restore defog environment from Drive tarball ──────────────────────
# Extracts the conda-pack tarball and fixes hardcoded absolute paths.
# The repo is not cloned yet at this point, so the restore logic is inline
# rather than delegating to colab_restore_env.sh.
# Runtime: ~4-6 min (Drive read + tar extraction + conda-unpack).
set -e

TARBALL='/content/drive/MyDrive/DeFoGColab/defog_env.tar.gz'
ENV_DIR='/content/miniconda3/envs/defog'

if [ ! -f "$TARBALL" ]; then
    echo "ERROR: $TARBALL not found. Run 01_env_setup.ipynb first." >&2
    exit 1
fi

echo "Tarball size: $(du -sh $TARBALL | cut -f1)"
echo "Extracting..."
mkdir -p "$ENV_DIR"
tar -xzf "$TARBALL" -C "$ENV_DIR"
echo "Running conda-unpack..."
"$ENV_DIR/bin/conda-unpack"
echo "Environment restored."
"$ENV_DIR/bin/python" --version

In [ ]:
# ── Cell 4: Configure — set your GitHub repo URL here ─────────────────────────
GITHUB_REPO = 'https://github.com/<YOUR_USERNAME>/DeFoGPlus.git'
REPO_DIR    = '/content/DeFoGPlus'

print(f'Will clone: {GITHUB_REPO}')

In [ ]:
%%bash -s "$GITHUB_REPO" "$REPO_DIR"
# ── Cell 5: Clone the DeFoG repository ───────────────────────────────────────
GITHUB_REPO="$1"
REPO_DIR="$2"
set -e
if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo already present — pulling latest."
    git -C "$REPO_DIR" pull
else
    git clone "$GITHUB_REPO" "$REPO_DIR"
fi
echo "Repo ready."
git -C "$REPO_DIR" log --oneline -3

In [ ]:
%%bash -s "$REPO_DIR"
# ── Cell 6: Wire Drive paths via symlinks ─────────────────────────────────────
#
# checkpoints symlink:  src/checkpoints/ → Drive/DeFoGColab/checkpoints/
#   Every checkpoint write by PyTorch Lightning goes directly to Drive.
#   Checkpoints survive session disconnects automatically.
#
# data symlink:         DeFoGPlus/data/  → Drive/DeFoGColab/data/
#   QM9 raw data downloaded on the first run is persisted to Drive and
#   reused in future sessions without re-downloading.
#   The precomputed .pt files are also visible through this symlink.
REPO_DIR="$1"
set -e

DRIVE_BASE='/content/drive/MyDrive/DeFoGColab'
mkdir -p "$DRIVE_BASE/checkpoints" "$DRIVE_BASE/data/qm9"

# checkpoints symlink (inside src/)
CKPT_LINK="$REPO_DIR/src/checkpoints"
if [ -L "$CKPT_LINK" ]; then
    rm "$CKPT_LINK"
fi
ln -s "$DRIVE_BASE/checkpoints" "$CKPT_LINK"
echo "Symlink: $CKPT_LINK → $DRIVE_BASE/checkpoints"

# data symlink (at repo root level)
DATA_LINK="$REPO_DIR/data"
if [ -L "$DATA_LINK" ] || [ -d "$DATA_LINK" ]; then
    rm -rf "$DATA_LINK"
fi
ln -s "$DRIVE_BASE/data" "$DATA_LINK"
echo "Symlink: $DATA_LINK → $DRIVE_BASE/data"

echo "Symlinks ready."

In [ ]:
# ── Cell 7: Full environment verification ─────────────────────────────────────
import subprocess, os

PYTHON     = '/content/miniconda3/envs/defog/bin/python'
REPO_DIR   = '/content/DeFoGPlus'
DRIVE_BASE = '/content/drive/MyDrive/DeFoGColab'
LIBGOMP    = '/content/miniconda3/envs/defog/lib/libgomp.so.1'

# LD_PRELOAD forces conda's newer libgomp to load before torch's bundled one;
# without it, graph_tool fails with "GOMP_5.0 not found" when imported after torch.
# PYTHONPATH lets `from src import utils` resolve when scripts run from src/.
BASE_ENV = {
    **os.environ,
    'MPLBACKEND': 'Agg',
    'LD_PRELOAD': LIBGOMP,
    'PYTHONPATH': REPO_DIR,
}

# ── 1. Python imports
check = """
import graph_tool, torch, rdkit, torch_geometric, pytorch_lightning, hydra, wandb, imageio
print('graph_tool       :', graph_tool.__version__)
print('torch            :', torch.__version__)
print('CUDA available   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU              :', torch.cuda.get_device_name(0))
print('torch_geometric  :', torch_geometric.__version__)
print('pytorch_lightning:', pytorch_lightning.__version__)
print('imageio          :', imageio.__version__)
"""
r = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True, env=BASE_ENV)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('Import check failed.')

# ── 2. W&B auth
r2 = subprocess.run(
    [PYTHON, '-m', 'wandb', 'whoami'],
    capture_output=True, text=True, env=BASE_ENV,
)
print('W&B identity:', r2.stdout.strip() or '(not logged in)')

# ── 3. Precomputed .pt file
pt_file = f'{REPO_DIR}/data/qm9/spminer_motif_marginals.pt'
status = '✓' if os.path.isfile(pt_file) else '✗  MISSING'
print(f'  {status}  {pt_file}')

# ── 4. Symlinks
for link, target in [
    (f'{REPO_DIR}/src/checkpoints', f'{DRIVE_BASE}/checkpoints'),
    (f'{REPO_DIR}/data',            f'{DRIVE_BASE}/data'),
]:
    ok = os.path.islink(link) and os.readlink(link) == target
    print(f'  {"✓" if ok else "✗"} symlink  {link}')

print('\nEnvironment ready — proceeding to ablation.')

---
## Marginal Ablation (QM9)

Runs two training + testing experiments back-to-back to isolate the effect
of the initial noise distribution on generation quality:

| Run | Transition | Marginals source |
|-----|-----------|------------------|
| `qm9_ablation_marginal` | `marginal` | Dataset type-frequency statistics |
| `qm9_ablation_spminer`  | `loaded_marginal` | SPMiner-mined motif marginals |

Each run: 100 epochs training → test with last checkpoint → results logged to W&B.  
Checkpoints are saved directly to Google Drive via the symlink set up above.

In [ ]:
# ── Cell 9: Paths, environment, and helper functions ──────────────────────────
import glob

# Paths and BASE_ENV (with LD_PRELOAD + PYTHONPATH) come from Cell 7.
SRC      = f'{REPO_DIR}/src'
DATA_QM9 = f'{REPO_DIR}/data/qm9'
RUN_ENV  = BASE_ENV

# ── Shared Hydra overrides for both runs ──────────────────────────────────────
# check_val_every_n_epochs=20  → validate at epochs 20, 40, 60, 80, 100
# sample_every_val=1           → checkpoint saved at every validation
# samples_to_generate=128      → small mid-training sample; keeps val fast
# final_model_samples_to_generate=512 → test budget
COMMON = [
    '+experiment=qm9_no_h',
    'hydra.job.chdir=False',
    'train.n_epochs=100',
    'general.check_val_every_n_epochs=20',
    'general.sample_every_val=1',
    'general.samples_to_generate=128',
    'general.final_model_samples_to_generate=512',
]


def run_live(cmd: list, cwd: str = SRC):
    """Run cmd, streaming stdout+stderr line by line.
    Raises RuntimeError on non-zero exit.
    """
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, cwd=cwd, env=RUN_ENV,
    )
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Process exited with code {proc.returncode}')


def train_and_test(name: str, extra: list):
    """Train for 100 epochs then test with the last checkpoint.

    Args:
        name:  W&B run name and checkpoint subdirectory name.
        extra: Run-specific Hydra overrides
               (e.g. model.transition=..., model.motif_marginals_path=...).
    """
    base_args = COMMON + [f'general.name={name}'] + extra

    print(f'\n{"╔" + "═"*58 + "╗"}')
    print(f'  TRAIN  →  {name}')
    print(f'{"╚" + "═"*58 + "╝"}\n')
    run_live([PYTHON, 'main.py'] + base_args)

    # src/checkpoints/ is symlinked to Drive — checkpoints persist across sessions.
    ckpt_dir = f'{SRC}/checkpoints/{name}'
    ckpts = sorted(glob.glob(f'{ckpt_dir}/*.ckpt'), key=os.path.getmtime)
    if not ckpts:
        raise RuntimeError(f'No checkpoint found in {ckpt_dir}.')
    ckpt = ckpts[-1]
    print(f'\nLast checkpoint: {ckpt}')

    print(f'\n{"╔" + "═"*58 + "╗"}')
    print(f'  TEST   →  {name}')
    print(f'{"╚" + "═"*58 + "╝"}\n')
    run_live([PYTHON, 'main.py'] + base_args + [f'general.test_only={ckpt}'])

    print(f'\n✓  {name} complete.  Checkpoint: {ckpt}')
    return ckpt


print('Helpers loaded.')
print(f'SRC      : {SRC}')
print(f'DATA_QM9 : {DATA_QM9}')

In [ ]:
# ── Cell 10: Pre-flight checks ────────────────────────────────────────────────
import torch

errors = []

# GPU
if torch.cuda.is_available():
    print(f'✓  GPU: {torch.cuda.get_device_name(0)}')
else:
    errors.append('No CUDA GPU detected — training will be extremely slow.')

# SPMiner marginals .pt file
spminer_pt = f'{DATA_QM9}/spminer_motif_marginals.pt'
if os.path.isfile(spminer_pt):
    m = torch.load(spminer_pt, map_location='cpu', weights_only=True)
    print(f'✓  SPMiner marginals: X={list(m["X"].shape)}  E={list(m["E"].shape)}')
else:
    errors.append(f'Missing: {spminer_pt}')

# Checkpoint symlink
ckpt_link   = f'{SRC}/checkpoints'
ckpt_target = f'{DRIVE_BASE}/checkpoints'
if os.path.islink(ckpt_link) and os.readlink(ckpt_link) == ckpt_target:
    print(f'✓  Checkpoint symlink → {ckpt_target}')
else:
    errors.append('Checkpoint symlink missing or wrong. Re-run Cell 6.')

# W&B key
if os.environ.get('WANDB_API_KEY', ''):
    print('✓  WANDB_API_KEY is set.')
else:
    print('⚠  WANDB_API_KEY not set — runs will not be logged to W&B.')

if errors:
    for e in errors:
        print(f'✗  {e}')
    raise RuntimeError('Pre-flight failed. Fix the issues above before continuing.')
else:
    print('\nAll checks passed — ready to train.')

In [ ]:
# ── Cell 11: Run 1 — unedited marginal distribution ───────────────────────────
#
# Baseline: the initial noise z_T is sampled from the dataset's empirical
# type-frequency marginals (no motif editing of any kind).
# This is the standard DeFoG marginal transition.

ckpt_marginal = train_and_test(
    name  = 'qm9_ablation_marginal',
    extra = ['model.transition=marginal'],
)

In [ ]:
# ── Cell 12: Run 2 — SPMiner motif-edited marginal distribution ───────────────
#
# The initial noise z_T is sampled from marginals that were estimated from
# QM9 graphs after injecting the top-1 SPMiner-mined motif (a 3-node chain).
# The only change relative to Run 1 is the rate matrix; the model architecture,
# training loop, and graph size are all identical.

SPMINER_PT = f'{DATA_QM9}/spminer_motif_marginals.pt'

ckpt_spminer = train_and_test(
    name  = 'qm9_ablation_spminer',
    extra = [
        'model.transition=loaded_marginal',
        f'model.motif_marginals_path={SPMINER_PT}',
    ],
)

In [ ]:
# ── Cell 13: Summary ──────────────────────────────────────────────────────────
print('=' * 60)
print('  Ablation complete')
print('=' * 60)
print()
rows = [
    ('qm9_ablation_marginal', 'marginal',        ckpt_marginal),
    ('qm9_ablation_spminer',  'loaded_marginal', ckpt_spminer),
]
for run_name, transition, ckpt in rows:
    print(f'Run        : {run_name}')
    print(f'Transition : {transition}')
    print(f'Checkpoint : {ckpt}')
    print()

print('Results are logged to W&B under the run names above.')
print('Checkpoints are persisted at:')
print(f'  {DRIVE_BASE}/checkpoints/')